In [4]:
"""
PC-structure diagnostic: silhouette + ARI for cell_type / sex / condition
=========================================================================
PD vs HC, n=155 (102 PD = ES+LS, 53 HC).

For each VST matrix (class-aware, stratum-aware):
  1. PCA on samples x genes; take PC1-PC50.
  2. For each label (condition, sex, cell_type):
       a. Silhouette score on PC1-50 with that label (Euclidean).
       b. KMeans(k = n_unique_labels) on PC1-50; ARI between KMeans clusters
          and the metadata labels.
  3. Tabulate and plot.

Interpretation
--------------
* Silhouette > 0.5 = strong cluster separation along that label;
  0.25-0.5 = moderate; < 0.25 = weak / overlapping.
* ARI > 0.5 = the unsupervised KMeans recovers the label well;
  0.2-0.5 = partial; < 0.2 = leading PCs do not encode that label.
* For a typical sorted-cell RNA-seq study, expect cell_type to dominate
  PC1-50 (high silhouette + high ARI), sex to be weak unless sex-chromosome
  genes are kept, and condition (PD vs HC) to be the smallest signal —
  which is exactly the diagnostic question for this pipeline.
"""

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
PDHC_DIR     = PIPELINE_DIR / "PDHC new"
META_PATH    = PIPELINE_DIR / "PDvsHC" / "Meta_data_PDHC.csv"
VST_DIR      = PDHC_DIR / "Outputs" / "VST"
OUTPUT_DIR   = PDHC_DIR / "Outputs" / "PC_structure_metrics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = [
    {"label": "class",   "vst": VST_DIR / "vst_class_aware_PDHC.csv"},
    {"label": "stratum", "vst": VST_DIR / "vst_stratum_aware_PDHC.csv"},
]

LABEL_COLS = ["condition", "cell_type", "sex"]
N_PCS      = 50
SEED       = 42

PALETTE = {"class": "#4C8DAE", "stratum": "#B85450"}


# ------------------------------- Helpers ------------------------------------
def load_vst(path: Path) -> pd.DataFrame:
    """Load gene x sample VST CSV; transpose if it landed samples x genes."""
    df = pd.read_csv(path, index_col=0)
    looks_like_genes = df.index.astype(str).str.startswith("ENSG").any()
    if not looks_like_genes:
        df = df.T
    return df  # genes x samples


def compute_metrics(X_pcs: np.ndarray, labels: np.ndarray,
                    n_clusters: int, seed: int) -> dict:
    """Silhouette + ARI on PC1-50 for a given label vector."""
    if len(np.unique(labels)) < 2:
        return {"silhouette": np.nan,
                "ari":        np.nan,
                "n_clusters_kmeans": n_clusters}

    sil = silhouette_score(X_pcs, labels, metric="euclidean")

    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed)
    clust = km.fit_predict(X_pcs)
    ari = adjusted_rand_score(labels, clust)

    return {"silhouette": float(sil),
            "ari":        float(ari),
            "n_clusters_kmeans": n_clusters}


def run_one(label: str, vst_path: Path, meta_full: pd.DataFrame) -> list:
    if not vst_path.exists():
        print(f"[{label}] skip - file not found: {vst_path}")
        return []

    print(f"\n=== {label}-aware ===")
    vst = load_vst(vst_path)
    sample_ids = vst.columns.tolist()
    X = vst.T.values.astype(np.float64)
    print(f"  Matrix: {X.shape[0]} samples x {X.shape[1]} genes")

    meta = meta_full.copy()
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    common = [s for s in sample_ids if s in meta.index]
    if len(common) != len(sample_ids):
        print(f"  [WARN] {len(sample_ids) - len(common)} samples missing in metadata")
    meta = meta.loc[common]
    keep_idx = [sample_ids.index(s) for s in common]
    X = X[keep_idx]

    pca = PCA(n_components=min(N_PCS, X.shape[0] - 1), random_state=SEED)
    X_pcs = pca.fit_transform(X)
    cum = pca.explained_variance_ratio_.cumsum()
    print(f"  PC1-{X_pcs.shape[1]} cumulative variance: {cum[-1]:.1%}")

    rows = []
    for col in LABEL_COLS:
        if col not in meta.columns:
            print(f"  [WARN] column '{col}' missing in metadata; skipped.")
            continue
        labels = meta[col].astype(str).values
        n_uniq = len(np.unique(labels))
        m = compute_metrics(X_pcs, labels, n_clusters=n_uniq, seed=SEED)
        rows.append({
            "matrix":               label,
            "label":                col,
            "n_classes":            n_uniq,
            "n_clusters_kmeans":    m["n_clusters_kmeans"],
            "silhouette":           m["silhouette"],
            "ari":                  m["ari"],
            "var_pc1_to_pc50_cum":   float(cum[-1]),
        })
        print(f"  {col:10s} n={n_uniq:2d}  silhouette={m['silhouette']:+.3f}  "
              f"ARI={m['ari']:+.3f}")
    return rows


# ------------------------------- Plot ---------------------------------------
def plot_results(df: pd.DataFrame, out_path: Path):
    long_df = df.melt(
        id_vars=["matrix", "label", "n_classes"],
        value_vars=["silhouette", "ari"],
        var_name="metric", value_name="value",
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=False)
    titles = {"silhouette": "Silhouette score (PC1-50)",
              "ari":        "Adjusted Rand Index (KMeans on PC1-50 vs label)"}
    benchmarks = {"silhouette": [(0.25, "weak"), (0.5, "strong")],
                  "ari":        [(0.20, "weak"), (0.5, "strong")]}

    for ax, metric in zip(axes, ["silhouette", "ari"]):
        sub = long_df[long_df["metric"] == metric]
        sns.barplot(
            data=sub, x="label", y="value", hue="matrix",
            order=LABEL_COLS, hue_order=["class", "stratum"],
            palette=PALETTE, edgecolor="white", linewidth=0.6, ax=ax,
        )
        for container in ax.containers:
            ax.bar_label(container, fmt="%+.2f", padding=2, fontsize=9)
        ax.axhline(0, color="black", linewidth=0.6)
        for level, lab in benchmarks[metric]:
            ax.axhline(level, color="gray", linestyle=":", linewidth=0.8)
            ax.text(0.0, level + 0.012, lab, color="gray",
                    fontsize=8, ha="left", va="bottom")
        ax.set_title(titles[metric], fontweight="bold")
        ax.set_ylabel("Score")
        ax.set_xlabel("")
        ax.legend(title="Filter", loc="upper right")

    fig.suptitle(
        "PC1-5 structure vs metadata labels - PD vs HC\n"
        "(higher = label aligns with leading variance directions)",
        fontweight="bold")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


# ------------------------------- Main ---------------------------------------
def main():
    if not META_PATH.exists():
        raise FileNotFoundError(f"Metadata not found: {META_PATH}")
    meta = pd.read_csv(META_PATH)

    all_rows = []
    for ds in DATASETS:
        all_rows.extend(run_one(ds["label"], ds["vst"], meta))

    if not all_rows:
        print("No results computed - input files missing.")
        return

    df = pd.DataFrame(all_rows)
    csv_out = OUTPUT_DIR / "pc_structure_metrics_PDHC.csv"
    df.to_csv(csv_out, index=False)
    plot_results(df, OUTPUT_DIR / "pc_structure_metrics_PDHC.png")

    print("\n" + "=" * 70)
    print("Summary table (PD vs HC):")
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"\nResults  : {csv_out}")
    print(f"Bar chart: {OUTPUT_DIR / 'pc_structure_metrics_PDHC.png'}")


if __name__ == "__main__":
    main()



=== class-aware ===
  Matrix: 155 samples x 12103 genes
  PC1-50 cumulative variance: 77.2%
  condition  n= 2  silhouette=+0.005  ARI=-0.010
  cell_type  n= 3  silhouette=+0.142  ARI=+0.457
  sex        n= 2  silhouette=+0.003  ARI=-0.014

=== stratum-aware ===
  Matrix: 155 samples x 7488 genes
  PC1-50 cumulative variance: 80.0%
  condition  n= 2  silhouette=+0.006  ARI=+0.012
  cell_type  n= 3  silhouette=+0.096  ARI=+0.073
  sex        n= 2  silhouette=+0.002  ARI=+0.044

Summary table (PD vs HC):
 matrix     label  n_classes  n_clusters_kmeans  silhouette     ari  var_pc1_to_pc50_cum
  class condition          2                  2      0.0052 -0.0096               0.7721
  class cell_type          3                  3      0.1422  0.4573               0.7721
  class       sex          2                  2      0.0031 -0.0137               0.7721
stratum condition          2                  2      0.0064  0.0117               0.8004
stratum cell_type          3                  3